In [1]:
#@title 1. Settings and helpers

BACKEND      = "ollama"     # "ollama" (works on T4) or "vllm" (L4/A100 only)
CTX          = 65536
OLLAMA_MODEL = "qwen3:4b-instruct-2507-q4_K_M"
VLLM_MODEL   = "Qwen/Qwen3-4B-Instruct-2507"
GW_PORT      = 18789        # OpenClaw gateway + Control UI
EXPOSE_MODEL = True         # True = also give the raw model endpoint its own public link

import os, re, json, time, socket, shutil, secrets, pathlib, subprocess, requests
from urllib.parse import urlsplit, quote

ip = get_ipython()
NODE_DIR = "/opt/node24"
os.environ["PATH"] = f"{NODE_DIR}/bin:/usr/local/bin:/root/.local/bin:" + os.environ["PATH"]
os.environ["OLLAMA_API_KEY"] = "ollama-local"   # tells OpenClaw a local Ollama is available

CREDS_FILE = pathlib.Path("/content/openclaw_creds.json")

if CREDS_FILE.exists():
    CREDS = json.loads(CREDS_FILE.read_text())
else:
    CREDS = {"gateway_token": secrets.token_urlsafe(32), "model_key": secrets.token_urlsafe(16),
             "term_user": "admin", "term_pass": secrets.token_urlsafe(10)}

def save_creds(): CREDS_FILE.write_text(json.dumps(CREDS))

save_creds()

TUNNELS = globals().get("TUNNELS", {})
URLS    = globals().get("URLS", {})
PROCS   = globals().get("PROCS", {})

def stop(name):
    p = PROCS.get(name)
    if p and p.poll() is None:
        p.terminate()

        try: p.wait(10)
        except Exception: p.kill()

def wait_for(url, headers=None, timeout=900, proc=None, log=None):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            if requests.get(url, headers=headers, timeout=3).ok: return
        except Exception: pass

        if proc is not None and proc.poll() is not None:
            print(open(log).read()[-4000:] if log else "")
            raise RuntimeError(f"Process behind {url} exited, see log above")

        time.sleep(5)
    raise TimeoutError(f"{url} not up after {timeout}s" + (f", check {log}" if log else ""))

def wait_port(port, timeout=180, proc=None, log=None):
    t0 = time.time()

    while time.time() - t0 < timeout:
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", port)) == 0: return

        if proc is not None and proc.poll() is not None:
            print(open(log).read()[-4000:] if log else "")

            raise RuntimeError(f"Process for port {port} exited, see log above")

        time.sleep(2)
    raise TimeoutError(f"Nothing listening on port {port}" + (f", check {log}" if log else ""))

def free_port():
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0)); return s.getsockname()[1]

def tunnel(name, port, host_header=None):
    old = TUNNELS.get(name)

    if old and old.poll() is None:
        old.terminate(); old.wait()

    log = f"/content/cf_{name}.log"
    args = ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{port}"]

    if host_header: args += ["--http-host-header", host_header]

    proc = subprocess.Popen(args, stdout=open(log, "w"), stderr=subprocess.STDOUT)
    TUNNELS[name] = proc

    for _ in range(45):
        found = [u for u in re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log).read())
                 if not u.startswith("https://api.")]
        if found: return found[0]
        if proc.poll() is not None: break
        time.sleep(2)
    raise RuntimeError(f"Tunnel '{name}' failed:\n" + open(log).read()[-2000:])

ip.system("nvidia-smi --query-gpu=name,memory.total --format=csv")

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
#@title 2. Model backend

if BACKEND == "ollama":
    if not shutil.which("ollama"):
        ip.system("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null")
        ip.system("curl -fsSL https://ollama.com/install.sh | sh")

    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Ollama already running")
    except Exception:
        env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_CONTEXT_LENGTH=str(CTX),
                   OLLAMA_FLASH_ATTENTION="1", OLLAMA_KV_CACHE_TYPE="q8_0", OLLAMA_NUM_PARALLEL="1")
        PROCS["ollama"] = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"),
                                           stderr=subprocess.STDOUT, env=env)
        wait_for("http://127.0.0.1:11434/api/tags", timeout=60, proc=PROCS["ollama"], log="/content/ollama.log")

    MODEL_ID = OLLAMA_MODEL
    ip.system(f"ollama pull {MODEL_ID}")
    MODEL_BASE, MODEL_KEY = "http://127.0.0.1:11434", "ollama-local"   # native API, no /v1

elif BACKEND == "vllm":
    name, cap = [x.strip() for x in subprocess.run(
        "nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader",
        shell=True, capture_output=True, text=True).stdout.split(",")]
    if float(cap) < 8.0:
        raise RuntimeError(f"{name} is too small for vLLM + 64K context. Set BACKEND='ollama'.")

    if not shutil.which("vllm"):
        ip.system("pip install -q vllm")

    MODEL_ID, MODEL_KEY = "qwen3-4b", CREDS["model_key"]
    subprocess.run(["pkill", "-f", "vllm serve"])
    PROCS["vllm"] = subprocess.Popen(
        ["vllm", "serve", VLLM_MODEL, "--served-model-name", MODEL_ID, "--port", "8000",
         "--max-model-len", str(CTX), "--gpu-memory-utilization", "0.92", "--enforce-eager",
         "--enable-auto-tool-choice", "--tool-call-parser", "hermes", "--api-key", MODEL_KEY],
        stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)
    wait_for("http://127.0.0.1:8000/health", timeout=1800, proc=PROCS["vllm"], log="/content/vllm.log")
    MODEL_BASE = "http://127.0.0.1:8000/v1"
else:
    raise ValueError("BACKEND must be 'ollama' or 'vllm'")

print(f"Backend: {BACKEND} | model: {MODEL_ID} | url: {MODEL_BASE}")


Backend: ollama | model: qwen3:4b-instruct-2507-q4_K_M | url: http://127.0.0.1:11434


In [3]:
#@title 3. Install Node 24, OpenClaw, cloudflared, ttyd and tmux

if not os.path.exists(f"{NODE_DIR}/bin/node"):
    idx = requests.get("https://nodejs.org/dist/latest-v24.x/", timeout=30).text
    fname = re.search(r"node-v24\.\d+\.\d+-linux-x64\.tar\.xz", idx).group(0)
    ip.system(f"mkdir -p {NODE_DIR} && curl -fsSL https://nodejs.org/dist/latest-v24.x/{fname}"
              f" | tar -xJ -C {NODE_DIR} --strip-components=1")

if not os.path.exists(f"{NODE_DIR}/bin/openclaw"):
    ip.system(f"{NODE_DIR}/bin/npm install -g openclaw@latest --loglevel=error")

for b in ("node", "npm", "npx", "openclaw"):
    if os.path.exists(f"{NODE_DIR}/bin/{b}"):
        os.system(f"ln -sf {NODE_DIR}/bin/{b} /usr/local/bin/{b}")

if not shutil.which("cloudflared"):
    ip.system("wget -q -O /usr/local/bin/cloudflared "
              "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
              " && chmod +x /usr/local/bin/cloudflared")

if not shutil.which("ttyd"):
    ip.system("wget -q -O /usr/local/bin/ttyd "
              "https://github.com/tsl0922/ttyd/releases/latest/download/ttyd.x86_64 && chmod +x /usr/local/bin/ttyd")

if not shutil.which("tmux"):
    ip.system("apt-get -qq install -y tmux > /dev/null")

ip.system("node -v && openclaw --version")

v24.21.0
OpenClaw 2026.10.2 (b8952d4)


In [4]:
#@title 4. Model Public URL

MODEL_PORT = 11434 if BACKEND == "ollama" else 8000

if EXPOSE_MODEL:
    URLS["model"] = tunnel("oc_model", MODEL_PORT,
                           host_header=f"localhost:{MODEL_PORT}" if BACKEND == "ollama" else None)
    time.sleep(10)

    hdrs = {"Authorization": f"Bearer {MODEL_KEY}"} if BACKEND == "vllm" else {}

    try:
        r = requests.get(URLS["model"] + "/v1/models", headers=hdrs, timeout=30)
        print("Check:", r.status_code, [m["id"] for m in r.json().get("data", [])])
    except Exception as e:
        print("Tunnel is up but the check failed (try again in a few seconds):", e)

    print("Model URL (OpenAI-compatible):", URLS["model"] + "/v1")

    if BACKEND == "ollama":
        print("Model URL (native Ollama):    ", URLS["model"])

else:
    old = TUNNELS.get("oc_model")

    if old and old.poll() is None:
        old.terminate()
    URLS.pop("model", None)
    print("Model endpoint not exposed (EXPOSE_MODEL = False)")

Check: 200 ['qwen3:4b-instruct-2507-q4_K_M']
Model URL (OpenAI-compatible): https://contrary-criteria-nancy-rome.trycloudflare.com/v1
Model URL (native Ollama):     https://contrary-criteria-nancy-rome.trycloudflare.com


In [5]:
#@title 5. Configure OpenClaw, start the gateway, and create the Control UI link

URLS["control"] = tunnel("oc_control", GW_PORT)
print("Control UI tunnel:", URLS["control"])

if BACKEND == "ollama":
    auth = ["--auth-choice", "ollama", "--custom-base-url", MODEL_BASE, "--custom-model-id", MODEL_ID]
else:
    auth = ["--auth-choice", "custom-api-key", "--custom-base-url", MODEL_BASE,
            "--custom-model-id", MODEL_ID, "--custom-api-key", MODEL_KEY,
            "--secret-input-mode", "plaintext", "--custom-compatibility", "openai"]

stop("gateway")
subprocess.run(["pkill", "-f", "openclaw gateway run"])
time.sleep(2)

cfg_file = pathlib.Path.home() / ".openclaw" / "openclaw.json"
if CREDS.get("onboarded_for") != f"{BACKEND}:{MODEL_ID}" or not cfg_file.exists():
    r = subprocess.run(["openclaw", "onboard", "--non-interactive", "--accept-risk", "--skip-health", *auth,
                        "--gateway-port", str(GW_PORT), "--gateway-bind", "loopback",
                        "--gateway-auth", "token", "--gateway-token", CREDS["gateway_token"]],
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if r.returncode != 0:
        raise RuntimeError("Onboarding failed, see output above")
    CREDS["onboarded_for"] = f"{BACKEND}:{MODEL_ID}"; save_creds()

ip.system(f"openclaw config set gateway.publicOrigin {URLS['control']}")

PROCS["gateway"] = subprocess.Popen(["openclaw", "gateway", "run"],
                                    stdout=open("/content/openclaw_gateway.log", "w"), stderr=subprocess.STDOUT)
wait_port(GW_PORT, timeout=180, proc=PROCS["gateway"], log="/content/openclaw_gateway.log")

time.sleep(5)
ip.system("openclaw health")
ip.system("openclaw models status")

Control UI tunnel: https://trans-tap-berry-situation.trycloudflare.com
│
◇  

OpenClaw 2026.10.2 (b8952d4)
Self-hosted, self-updating, self-aware (just kidding... unless?).

Updated gateway.publicOrigin. Change will apply without restarting the gateway.

OpenClaw 2026.10.2 (b8952d4)
I'm not trapped in this container with you—you're trapped in here with me.

│

[
3
5
m
◒

[
3
9m
 
 

[
9
1
m
C
h
e
c
k
i
n
g
 
g
a
t
e
w
a
y
 
h
e
a
l
t
h
…

[
3
9m◇  
Gateway event loop: ok max=38ms p99=38ms util=0.047 cpu=0.364
Gateway probe duration: 461ms
Agents: main (default)
Heartbeat interval: 30m (main)
Session store (main): /root/.openclaw/agents/main/agent/openclaw-agent.sqlite (0 entries)

OpenClaw 2026.10.2 (b8952d4)
Your .zshrc wishes it could do what I do.

Config        : ~/.openclaw/openclaw.json
Agent dir     : ~/.openclaw/agents/main/agent
Default       : ollama/qwen3:4b-instruct-2507-q4_K_M
Fallbacks (0) : -
Utility model : -
Image model   : -
Image fallbacks (0): -
Aliases (0)   : 

In [6]:
#@title 6. Trust local cloudflared as a proxy

cfg_file = pathlib.Path.home() / ".openclaw" / "openclaw.json"
ip.system("""openclaw config set gateway.trustedProxies '["127.0.0.1","::1"]'""")

try:
    cfg = json.loads(cfg_file.read_text())
    if cfg.get("gateway", {}).get("trustedProxies") != ["127.0.0.1", "::1"]:
        cfg.setdefault("gateway", {})["trustedProxies"] = ["127.0.0.1", "::1"]
        cfg_file.write_text(json.dumps(cfg, indent=2))
    print("trustedProxies:", json.loads(cfg_file.read_text())["gateway"]["trustedProxies"])
except Exception as e:
    print("Could not verify config file:", e)

stop("gateway")
subprocess.run(["pkill", "-f", "openclaw gateway run"])
time.sleep(2)
PROCS["gateway"] = subprocess.Popen(["openclaw", "gateway", "run"],
                                    stdout=open("/content/openclaw_gateway.log", "w"), stderr=subprocess.STDOUT)
wait_port(GW_PORT, timeout=180, proc=PROCS["gateway"], log="/content/openclaw_gateway.log")
time.sleep(5)
ip.system("openclaw health")

│
◇  

OpenClaw 2026.10.2 (b8952d4)
Home is wherever port 18789 is.

No change
trustedProxies: ['127.0.0.1', '::1']

OpenClaw 2026.10.2 (b8952d4)
I'm not saying your workflow is chaotic... I'm just bringing a linter and a helmet.

│
◇  
Gateway event loop: degraded for 5s reasons=cpu max=86ms p99=86ms util=0.206 cpu=1.236
Gateway probe duration: 5632ms
Agents: main (default)
Heartbeat interval: 30m (main)
Session store (main): /root/.openclaw/agents/main/agent/openclaw-agent.sqlite (0 entries)


In [7]:
#@title 7. One-time login link for your browser

r = subprocess.run(["openclaw", "dashboard", "--json"], capture_output=True, text=True, timeout=60)

try:
    data = json.loads(r.stdout[r.stdout.index("{"):])
    bu = data["browserUrl"]
    loc = urlsplit(bu)
    pub = URLS["control"]
    wss = "wss://" + urlsplit(pub).netloc
    link = bu.replace(f"{loc.scheme}://{loc.netloc}", pub, 1)
    for old in (f"ws://{loc.netloc}", f"wss://{loc.netloc}"):
        link = link.replace(quote(old, safe=""), quote(wss, safe="")).replace(old, wss)
    print("Open this within 10 minutes, in the browser you'll keep using (not a private window):\n")
    print(link)

except Exception:
    print(r.stdout, r.stderr)
    print(f"\nFallback: open {URLS['control']} , paste the gateway token from Cell 8, then use Cell 6.")

Open this within 10 minutes, in the browser you'll keep using (not a private window):

https://trans-tap-berry-situation.trycloudflare.com/#bootstrapToken=XZmLdze1fiJ0JKFzkyKEa96fbXXeMhP35LmenEjoB80&bootstrapProfile=owner&gatewayUrl=wss%3A%2F%2Ftrans-tap-berry-situation.trycloudflare.com


In [8]:
# ip.system("openclaw devices list")
# REQUEST_ID = "https://kilometers-bet-reuters-commercial.trycloudflare.com/#bootstrapToken=UTZY7PbghcPps2HPQfPBlCpAQQlD7jOS9sJsTbKZ-MM&bootstrapProfile=owner&gatewayUrl=wss%3A%2F%2Fkilometers-bet-reuters-commercial.trycloudflare.com"      # paste the request ID from the list above, then re-run this cell
# if REQUEST_ID:
#     ip.system(f"openclaw devices approve {REQUEST_ID}")

In [9]:
#@title 8. Browser terminal that opens the OpenClaw interface

tmux_conf = pathlib.Path("/root/.tmux.conf")

if not tmux_conf.exists():
    tmux_conf.write_text('set -g default-terminal "tmux-256color"\n'
                         'set -ga terminal-overrides ",xterm-256color:Tc"\n'
                         'set -g mouse on\nset -g history-limit 50000\n')

stop("oc_ttyd")
subprocess.run(["tmux", "kill-session", "-t", "openclaw"], capture_output=True)

OC_TERM_PORT = free_port()
PROCS["oc_ttyd"] = subprocess.Popen(
    ["ttyd", "-p", str(OC_TERM_PORT), "-W",
     "-c", f"{CREDS['term_user']}:{CREDS['term_pass']}",
     "-t", "fontSize=15", "-t", "titleFixed=OpenClaw",
     "tmux", "new", "-A", "-s", "openclaw", "bash -lc 'openclaw tui; exec bash'"],
    stdout=open("/content/oc_ttyd.log", "w"), stderr=subprocess.STDOUT,
    env=dict(os.environ, TERM="xterm-256color"))

wait_port(OC_TERM_PORT, timeout=30, proc=PROCS["oc_ttyd"], log="/content/oc_ttyd.log")

URLS["terminal"] = tunnel("oc_terminal", OC_TERM_PORT)
print("Terminal:", URLS["terminal"])

Terminal: https://goal-tahoe-william-void.trycloudflare.com


In [10]:
#@title 9. All links and logins

if URLS.get("model"):
    print(f"Model API   {URLS['model']}/v1")
    print(f"            model: {MODEL_ID}")
    print(f"            key:   {MODEL_KEY if BACKEND == 'vllm' else '(none: Ollama has no auth, any value works)'}")

print(f"""
Control UI  {URLS.get('control')}
            log in with the one-time link from Cell 5
            (or paste gateway token: {CREDS['gateway_token']})

Terminal    {URLS.get('terminal')}
            login: {CREDS['term_user']} / {CREDS['term_pass']}
            opens straight into `openclaw tui`
""")

Model API   https://contrary-criteria-nancy-rome.trycloudflare.com/v1
            model: qwen3:4b-instruct-2507-q4_K_M
            key:   (none: Ollama has no auth, any value works)

Control UI  https://trans-tap-berry-situation.trycloudflare.com
            log in with the one-time link from Cell 5
            (or paste gateway token: 6ykM2vVD_KoiTvl4-vkrE5F7sxNsiWueGIS0IJEsvY8)

Terminal    https://goal-tahoe-william-void.trycloudflare.com
            login: admin / u6debCqscuL1TQ
            opens straight into `openclaw tui`



In [ ]:
#@title 10. Run loop

while True:
    time.sleep(60)